In [0]:
%run ./config

In [0]:
import requests

SEC_HEADERS = {"User-Agent": dbutils.secrets.get(SECRET_SCOPE, "sec_user_agent")}

tickers = requests.get("https://www.sec.gov/files/company_tickers.json", headers=SEC_HEADERS, timeout=30)
print("ticker list:", tickers.status_code, "·", len(tickers.json()), "companies")

apple = requests.get("https://data.sec.gov/submissions/CIK0000320193.json", headers=SEC_HEADERS, timeout=30).json()
print("Apple:", apple["name"], "· CIK", apple["cik"], "· industry", apple["sic"], apple["sicDescription"])
print("former names:", apple.get("formerNames"))

In [0]:
import json, os, time
from datetime import date

SNAPSHOT = date.today().isoformat()
LANDING = f"/Volumes/{CATALOG}/bronze/raw/sec/snapshot={SNAPSHOT}"
os.makedirs(f"{LANDING}/submissions", exist_ok=True)

# 1) The SEC's full ticker → CIK list, saved as received
with open(f"{LANDING}/company_tickers.json", "w") as f:
    f.write(tickers.text)
sec_map = {v["ticker"].upper(): str(v["cik_str"]).zfill(10) for v in tickers.json().values()}

# 2) Our universe → CIK
universe = [r.ticker for r in spark.table(f"{CATALOG}.gold.ref_universe").select("ticker").collect()]
mapped = {t: sec_map[t] for t in universe if t in sec_map}
unmapped = sorted(set(universe) - set(mapped))

# 3) One profile per company (several tickers can share a CIK, e.g. two share classes)
downloaded = 0
for cik in sorted(set(mapped.values())):
    path = f"{LANDING}/submissions/CIK{cik}.json"
    if os.path.exists(path):
        continue
    resp = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=SEC_HEADERS, timeout=30)
    resp.raise_for_status()
    with open(path, "w") as f:
        f.write(resp.text)
    downloaded += 1
    time.sleep(0.15)                       # stay well under the SEC's 10 requests/second

print(f"Mapped {len(mapped)} of {len(universe)} tickers to a CIK · unmapped: {unmapped or 'none'}")
print(f"Profiles downloaded this run: {downloaded} · on disk: {len(os.listdir(f'{LANDING}/submissions'))}")

In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())

def lineage(df):
    return (df.withColumn("_source_system", F.lit("sec_edgar"))
              .withColumn("_snapshot", F.lit(SNAPSHOT))
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_run_id", F.lit(RUN_ID)))

# 1) Ticker list: the file is one JSON object of objects, so parse it, then make rows
raw_text = spark.read.text(f"{LANDING}/company_tickers.json", wholetext=True).first()[0]
ticker_rows = [(str(v["cik_str"]).zfill(10), v["ticker"], v["title"]) for v in json.loads(raw_text).values()]
lineage(spark.createDataFrame(ticker_rows, "cik string, ticker string, title string")) \
    .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{CATALOG}.bronze.sec_company_tickers")

# 2) Company profiles: read only the fields we need
profile_schema = """cik STRING, name STRING, entityType STRING, sic STRING, sicDescription STRING,
                    tickers ARRAY<STRING>, exchanges ARRAY<STRING>,
                    formerNames ARRAY<STRUCT<name: STRING, `from`: STRING, `to`: STRING>>"""
profiles = (spark.read.schema(profile_schema).option("multiLine", True).json(f"{LANDING}/submissions")
            .withColumn("_source_file", F.col("_metadata.file_path")))
lineage(profiles).write.mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.sec_submissions")

# 3) Gate: files → bronze
n_files = len(dbutils.fs.ls(f"{LANDING}/submissions"))
sub = spark.table(f"{CATALOG}.bronze.sec_submissions").filter(F.col("_run_id") == RUN_ID)
n_rows, n_missing = sub.count(), sub.filter("cik IS NULL OR name IS NULL").count()
n_tick = spark.table(f"{CATALOG}.bronze.sec_company_tickers").filter(F.col("_run_id") == RUN_ID).count()

def row(table, check, exp, act, ok):
    return (RUN_ID, "files_to_bronze", table, check, "error", str(exp), str(act),
            "PASS" if ok else "FAIL", datetime.now(timezone.utc))

results = [
    row("bronze.sec_company_tickers", "row_count_matches_file", len(ticker_rows), n_tick, n_tick == len(ticker_rows)),
    row("bronze.sec_submissions", "one_row_per_file", n_files, n_rows, n_rows == n_files),
    row("bronze.sec_submissions", "cik_and_name_present", 0, n_missing, n_missing == 0),
]
gate = spark.createDataFrame(results, spark.table(f"{CATALOG}.audit.gate_results").schema)
gate.write.mode("append").saveAsTable(f"{CATALOG}.audit.gate_results")
display(gate)
assert all(r[7] == "PASS" for r in results), "❌ Gate files_to_bronze (SEC) FAILED"

display(sub.select("cik", "name", "sic", "sicDescription", "tickers", F.size("formerNames").alias("former_names"))
           .orderBy(F.desc("former_names")).limit(10))